# Libraries setup

In [ ]:
#!pip install transformers accelerate sentencepiece torch rdkit

: 

# Zero Short Implementation

In [ ]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rdkit.Chem import MolFromSmiles
from rdkit import RDLogger

RDLogger.DisableLog('rdApp.*')

MODEL_NAME = "insilicomedicine/nach0_base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

model.eval()

# Official Nach0 preprocessing
atoms_tokens = [
    'Ag','Al','As','Au','B','Ba','Bi','Br','C','Ca',
    'Cd','Cl','Co','Cr','Cs','Cu','F','Fe','Ga','Gd',
    'Ge','H','Hg','I','In','K','Li','M','Mg','Mn',
    'Mo','N','Na','O','P','Pt','Ru','S','Sb','Sc',
    'Se','Si','Sn','V','W','Z','Zn','c','e','n','o','p','s'
]

atoms_tokens = sorted(atoms_tokens, key=lambda s: len(s), reverse=True)

SMI_REGEX_PATTERN = (
    r"(\[|\]|\(|\)|\.|=|#|-|\+|\\|\/|:|~|@|\?|>>?|\*|\$|\%[0-9]{2}|[0-9]|"
    + '|'.join(atoms_tokens) +
    ")"
)

regex = re.compile(SMI_REGEX_PATTERN)


def clean_output_sequence(output_sequence):

    return (
        output_sequence
        .replace('</s>', '')
        .replace('<sm_', '')
        .replace(' sm_', '')
        .replace('>', '')
        .strip()
    )


def add_special_symbols(text):

    output = []

    for word in text.split():

        tokens = [token for token in regex.findall(word)]

        if len(tokens) > 4 and (word == ''.join(tokens)) and MolFromSmiles(word):

            output.append(
                ''.join(['<sm_' + t + '>' for t in tokens])
            )

        else:
            output.append(word)

    return ' '.join(output)

# Generation function
def generate_response(prompt, max_tokens=10):

    processed_prompt = add_special_symbols(prompt)

    inputs = tokenizer(
        processed_prompt,
        return_tensors="pt",
        max_length=512
    )

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            temperature=0.6,
            max_new_tokens = max_tokens,
            do_sample=True,
            top_p=0.95,
            top_k=100
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    decoded = clean_output_sequence(decoded)
    return decoded

# Zero-shot prompting
def zero_shot_flavor_prediction(smiles):

    prompt = f"""
          Possible flavor labels: bitter, sour, sweet, umami

          Classify the flavor of the given molecule in one labels mentioned.
          Molecule: {smiles}
          Flavor:
      """

    output = generate_response(prompt)

    print(f"\nGenerated Output for {smiles}:")
    print(output)

    return output

# Test Cases
TEST_CASES = [
    {
        "smiles": "COC(=O)C(Cc1ccccc1)NC(=O)C(NC=O)C(CC=C(C)C=CC=C(C)C=CC1=C(C)CCCC1(C)C)C(=O)O",
        "expected": "Sweet"
    },
    {
        "smiles": "NC(=O)CC(N)C(=O)NC(CC(=O)O)C(=O)NC(Cc1ccccc1)C(=O)NCC(=O)O",
        "expected": "Sweet"
    },
    {
        "smiles": "O=C(O)c1cccc(CO)n1",
        "expected": "Sour"
    },
    {
        "smiles": "ON=CC1=CCSCC1",
        "expected": "Bitter"
    },
    {
        "smiles": "COc1cc(-c2cc(=O)c3c(O)cc(OC4OCC(O)C(O)C4O)c(O)c3o2)cc(O)c1O",
        "expected": "Sweet"
    },
    {
        "smiles": "CC1=CCC(C(C)C)=CC1",
        "expected": "Bitter"
    },
    {
        "smiles": "O=Cc1ccncc1O",
        "expected": "Sour"
    }
]

# Run all test cases
for idx, test in enumerate(TEST_CASES, start=1):

    print("\n" + "=" * 80)
    print(f"TEST CASE {idx}")
    print("=" * 80)

    print(f"SMILES     : {test['smiles']}")
    print(f"Expected   : {test['expected']}")

    prediction = zero_shot_flavor_prediction(test['smiles'])

    print(f"\nFinal Prediction : {prediction}")


In [ ]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rdkit.Chem import MolFromSmiles
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

# ── Model ────────────────────────────────────────────────────────────────────
MODEL_NAME = "insilicomedicine/nach0_base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.eval()

# ── Official Nach0 SMILES tokenizer ──────────────────────────────────────────
atoms_tokens = [
    'Ag','Al','As','Au','B','Ba','Bi','Br','C','Ca',
    'Cd','Cl','Co','Cr','Cs','Cu','F','Fe','Ga','Gd',
    'Ge','H','Hg','I','In','K','Li','M','Mg','Mn',
    'Mo','N','Na','O','P','Pt','Ru','S','Sb','Sc',
    'Se','Si','Sn','V','W','Z','Zn','c','e','n','o','p','s'
]
atoms_tokens = sorted(atoms_tokens, key=lambda s: len(s), reverse=True)
SMI_REGEX_PATTERN = (
    r"(\[|\]|\(|\)|\.|=|#|-|\+|\\|\/|:|~|@|\?|>>?|\*|\$|\%[0-9]{2}|[0-9]|"
    + '|'.join(atoms_tokens)
    + ")"
)
regex = re.compile(SMI_REGEX_PATTERN)


def clean_output_sequence(output_sequence):
    """Remove nach0-specific special token artifacts from decoded output."""
    return (
        output_sequence
        .replace('</s>', '')
        .replace('<sm_', '')
        .replace(' sm_', '')
        .replace('>', '')
        .strip()
    )


def add_special_symbols(text):
    """Replace valid SMILES tokens with nach0 special-token format."""
    output = []
    for word in text.split():
        tokens = [token for token in regex.findall(word)]
        if len(tokens) > 4 and (word == ''.join(tokens)) and MolFromSmiles(word):
            output.append(''.join(['<sm_' + t + '>' for t in tokens]))
        else:
            output.append(word)
    return ' '.join(output)


# ── Generation — matches official nach0 API exactly ──────────────────────────
def generate_response(prompt: str, max_length: int = 512) -> str:
    """
    prompt      : plain-text prompt (SMILES will be encoded by add_special_symbols)
    max_length  : total token budget for the *generated* sequence (default 512)
    """
    processed_prompt = add_special_symbols(prompt)

    # Official tokenizer call: padding="longest", truncation=True
    input_text_ids = tokenizer(
        processed_prompt,
        padding="longest",
        max_length=512,
        truncation=True,
        return_tensors="pt",
    )

    with torch.no_grad():
        # Official generate call — max_length, NOT max_new_tokens
        # temperature is intentionally omitted (not in official guide)
        generated_text_ids = model.generate(
            **input_text_ids,
            do_sample=True,
            top_k=100,
            top_p=0.95,
            max_length=max_length,
        )

    # Official decode: batch_decode with skip_special_tokens=True, then clean
    generated_text = tokenizer.batch_decode(
        generated_text_ids, skip_special_tokens=True
    )[0]
    return clean_output_sequence(generated_text)


# ── Flavor prediction ─────────────────────────────────────────────────────────
def zero_shot_flavor_prediction(smiles: str) -> str:
    """
    Build a flat single-line prompt so `add_special_symbols` can correctly
    detect the SMILES string as a standalone whitespace-separated token.
    """
    prompt = (
        "Possible flavor labels: bitter, sour, sweet, umami. "
        "Classify the flavor of the given molecule using one of the labels above. "
        f"Molecule: {smiles} "
        "Flavor:"
    )
    output = generate_response(prompt)
    return output


# ── Test cases ────────────────────────────────────────────────────────────────
TEST_CASES = [
    {"smiles": "COC(=O)C(Cc1ccccc1)NC(=O)C(NC=O)C(CC=C(C)C=CC=C(C)C=CC1=C(C)CCCC1(C)C)C(=O)O", "expected": "Sweet"},
    {"smiles": "NC(=O)CC(N)C(=O)NC(CC(=O)O)C(=O)NC(Cc1ccccc1)C(=O)NCC(=O)O",                   "expected": "Sweet"},
    {"smiles": "O=C(O)c1cccc(CO)n1",                                                              "expected": "Sour"},
    {"smiles": "ON=CC1=CCSCC1",                                                                   "expected": "Bitter"},
    {"smiles": "COc1cc(-c2cc(=O)c3c(O)cc(OC4OCC(O)C(O)C4O)c(O)c3o2)cc(O)c1O",                  "expected": "Sweet"},
    {"smiles": "CC1=CCC(C(C)C)=CC1",                                                             "expected": "Bitter"},
    {"smiles": "O=Cc1ccncc1O",                                                                    "expected": "Sour"},
]

for idx, test in enumerate(TEST_CASES, start=1):
    print("\n" + "=" * 80)
    print(f"TEST CASE {idx}")
    print("=" * 80)
    print(f"SMILES     : {test['smiles']}")
    print(f"Expected   : {test['expected']}")
    prediction = zero_shot_flavor_prediction(test['smiles'])
    print(f"Final Prediction : {prediction}")


print(f"\n{'='*80}")